# Bank marketing exploration and benchmark
Run from the project directory with the CSV at `data/bank-additional-full.csv`. This notebook checks training-period SQL and Pandas summaries, then evaluates logistic regression under a 100-call budget.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display

from bank_marketing.analysis import class_balance, missing_values, summarize_channels
from bank_marketing.benchmark import format_results, run_benchmark
from bank_marketing.data import load_campaign, prepare_xy, split_chronologically

## Load and split in chronological order

In [ ]:
data_dir = Path("data")
df = load_campaign(data_dir / "bank-additional-full.csv")
print(df.shape)

train, val, test = split_chronologically(df)
print(len(train), len(val), len(test))
assert len(train) + len(val) + len(test) == len(df)
print("Training subscription rate", train["y"].eq("yes").mean())
print("Validation subscription rate", val["y"].eq("yes").mean())

## Inspect the audited feature set
The model uses nine customer profile and prior-campaign fields. See the [feature audit](docs/feature_audit.md) for scoring-time assumptions and exclusions.

In [ ]:
X_train, y_train = prepare_xy(train)
X_val, y_val = prepare_xy(val)
assert X_val.columns.equals(X_train.columns)
assert X_val.index.equals(y_val.index)
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

## Query the training period

In [ ]:
with closing(sqlite3.connect(data_dir / "bank_marketing.db")) as conn:
    train.to_sql("campaign_train", conn, if_exists="replace", index=False)
    row_count = conn.execute("SELECT COUNT(*) FROM campaign_train").fetchone()[0]
    assert row_count == len(train)
    channel_summary = pd.read_sql_query(Path("sql/channel_summary.sql").read_text(), conn)
    job_summary = pd.read_sql_query(Path("sql/job_summary.sql").read_text(), conn)

assert channel_summary["n_records"].sum() == len(train)
display(channel_summary, job_summary)

## Reproduce the channel summary in Pandas

In [ ]:
pandas_summary = summarize_channels(train)
sql_channels = channel_summary.set_index("contact").sort_index()
pandas_channels = pandas_summary.set_index("contact").sort_index()
assert sql_channels.index.equals(pandas_channels.index)
for column in ["n_records", "n_subscribed"]:
    assert sql_channels[column].equals(pandas_channels[column])
np.testing.assert_allclose(sql_channels["subscription_rate"], pandas_channels["subscription_rate"])
np.testing.assert_allclose(pandas_channels["subscription_rate"],
                           pandas_channels["n_subscribed"] / pandas_channels["n_records"])
display(pandas_summary)

## Class balance, unknown values and prior contact

In [ ]:
display(class_balance(train), missing_values(train))
prior_contact_summary = train.loc[train["previous"].eq(0), ["pdays", "poutcome"]].value_counts()
display(prior_contact_summary)

In the training data, 27,829 records with zero prior contacts have `pdays = 999` and `poutcome = nonexistent`. This identifies the absence of a prior campaign contact. The literal string `unknown` marks unavailable information in fields such as `default`.

## Age and job distributions

In [ ]:
display(train["age"].describe())
display(train["job"].value_counts())

## Student subscription rate
Of 237 students in training, 33 subscribed (13.92%). The exercise used a rounded standard error of 2.2 percentage points and an approximate normal interval. Under the independent, representative sampling assumption, the resulting 95% interval for the population rate is 9.61% to 18.23%. This is a descriptive interval for the observed group. Selecting the highest-rate group adds uncertainty to comparisons across jobs.

## Evaluate the 100-call budget
Fit one logistic regression pipeline on the training period. Rank each later period by predicted probability. The random baseline is the expected number of subscribers in an equally sized random sample from the same period. This is a retrospective benchmark on historical contact records.

In [ ]:
results = run_benchmark(train, val, test)
print(format_results(results))